In [ ]:
import pandas as pd
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer
import torch
from tqdm.auto import tqdm
import os # Cần import os
import gc

print("--- SCRIPT ĐÁNH GIÁ (EVALUATION) ---")

# --- 1. Cấu hình ---
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"--- Đang chạy trên: {device} ---")

# Định nghĩa các mô hình VÀ CÁC INDEX ĐÃ ĐƯỢC XÂY SẴN
models_to_evaluate = {
    "Model_A_Baseline": {
        "model_path": "paraphrase-multilingual-mpnet-base-v2",
        "index_path": "./artifacts/index_Model A (Baseline).faiss"
    },
    "Model_B_Triplet_1Epoch": {
        "model_path": "./models/triplet-finetuned-model-v2/",
        "index_path": "./artifacts/medical_index_B.faiss"
    },
    "Model_C_MiniLM_Tuned": {
        "model_path": "./models/minilm-finetuned-v1/",
        "index_path": "./artifacts/index_Model C (MiniLM-tuned).faiss"
    },
    "Model_D_BERT_Tuned": {
        "model_path": "./models/bert-base-multilingual-cased-finetuned-v1/",
        "index_path": "./artifacts/index_Model D (BERT-tuned).faiss"
    },

    "Model_E_DistilUSE_Tuned": {
        "model_path": "./models/distiluse-base-multilingual-cased-v1-finetuned-v1/",
        "index_path": "./artifacts/index_Model E (distiluse-tuned).faiss"
    }
}

CORPUS_FILE = 'corpus_with_id.csv'
TEST_SET_FILE = 'test_set_new.csv'
TOP_K_SEARCH = 10 # Tìm Top 10

# --- 2. Tải Dữ liệu (Corpus và Test Set) ---
print("Đang tải dữ liệu...")
corpus_df = pd.read_csv(CORPUS_FILE)
# Tạo map từ index (0,1,2...) sang doc_id gốc
corpus_doc_ids = corpus_df['doc_id'].tolist()
index_to_id_map = {i: doc_id for i, doc_id in enumerate(corpus_doc_ids)}

test_set_df = pd.read_csv(TEST_SET_FILE)
test_set_df = test_set_df.dropna(subset=['query', 'ground_truth_doc_id'])
test_set_df['ground_truth_doc_id'] = test_set_df['ground_truth_doc_id'].astype(int)
print(f"Đã tải {len(test_set_df)} câu test.")


# --- 3. Hàm Tính toán (Giữ nguyên) ---
def calculate_metrics(retrieved_doc_ids, ground_truth_id, k_values=[1, 3, 5, 10]):
    metrics = {}
    retrieved_set = set(retrieved_doc_ids)
    for k in k_values:
        top_k_set = set(retrieved_doc_ids[:k])
        is_hit = 1.0 if ground_truth_id in top_k_set else 0.0
        metrics[f'Recall@{k}'] = is_hit
    reciprocal_rank = 0.0
    for rank, doc_id in enumerate(retrieved_doc_ids):
        if doc_id == ground_truth_id:
            reciprocal_rank = 1.0 / (rank + 1)
            break
    metrics['MRR'] = reciprocal_rank
    return metrics

# --- 4. Vòng lặp Đánh giá (Đã Cập nhật) ---
print("\n--- BẮT ĐẦU VÒNG LẶP ĐÁNH GIÁ (MODEL TOURNAMENT) ---")

all_results = {} # Lưu kết quả trung bình của các mô hình

for model_name, paths in models_to_evaluate.items():
    print(f"\n--- Đang đánh giá: {model_name} ---")
    
    try:
        # 4a. Tải Model (Chỉ để mã hóa query)
        print(f"Đang tải model: {paths['model_path']}")
        model = SentenceTransformer(paths['model_path'], device=device)
        
        # 4b. Tải Index (ĐÃ ĐƯỢC XÂY SẴN)
        print(f"Đang tải index: {paths['index_path']}")
        index = faiss.read_index(paths['index_path'])
        
        model_results = [] # Lưu kết quả của từng query
        
        # 4c. Chạy Vòng lặp Test Set
        for row in tqdm(test_set_df.itertuples(), total=len(test_set_df), desc=f"Testing {model_name}"):
            query = row.query
            ground_truth_id = int(row.ground_truth_doc_id)
            
            # Mã hóa query
            query_vec = model.encode([query])
            faiss.normalize_L2(query_vec)
            
            # Tìm kiếm (Rất nhanh)
            D, I = index.search(query_vec.astype(np.float32), TOP_K_SEARCH)
            
            # Chuyển index của FAISS (I[0]) sang doc_id GỐC
            retrieved_doc_ids = [index_to_id_map[i] for i in I[0]]
            
            # Tính toán
            metrics = calculate_metrics(retrieved_doc_ids, ground_truth_id)
            model_results.append(metrics)
        
        # 4d. Tính trung bình và lưu kết quả
        all_results[model_name] = pd.DataFrame(model_results).mean()

    except Exception as e:
        print(f"LỖI khi đánh giá {model_name}: {e}")
        
    finally:
        # Giải phóng VRAM
        del model, index
        gc.collect()
        if device == 'cuda':
            torch.cuda.empty_cache()

# --- 5. Hiển thị Kết quả Cuối cùng ---
print("\n--- 5. KẾT QUẢ CUỐI CÙNG (TOURNAMENT RESULTS) ---")

if all_results:
    final_comparison_df = pd.DataFrame(all_results).T # .T (Transpose) để tên model làm hàng
    
    # Thêm cột so sánh với Baseline
    if "Model_A_Baseline" in final_comparison_df.index:
        baseline_recall = final_comparison_df.loc["Model_A_Baseline", 'Recall@5']
        final_comparison_df['Improvement@5 (%)'] = (
            (final_comparison_df['Recall@5'] - baseline_recall) / baseline_recall
        ) * 100
    
    print(final_comparison_df.to_markdown(floatfmt=".4f"))
    
    # Quyết định
    best_model_name = final_comparison_df['Recall@5'].idxmax()
    print(f"\n--- QUYẾT ĐỊNH ---")
    print(f"GO: Mô hình tốt nhất là: {best_model_name}")
    print(f"   (Dựa trên Recall@5 cao nhất: {final_comparison_df.loc[best_model_name, 'Recall@5']:.4f})")
else:
    print("Không có kết quả nào để hiển thị.")

In [ ]:
import pandas as pd
from pyvi import ViTokenizer # Thư viện tách từ Tiếng Việt
from rank_bm25 import BM25Okapi
import pickle # Dùng để lưu lại index
import os

print("--- Script tạo Chỉ mục BM25 ---")

# --- 1. Cấu hình ---
CORPUS_FILE = 'corpus_with_id.csv'
OUTPUT_DIR = './artifacts/'
OUTPUT_BM25_FILE = os.path.join(OUTPUT_DIR, 'medical_index_B.bm25')

os.makedirs(OUTPUT_DIR, exist_ok=True)

# --- 2. Tải Dữ liệu Corpus ---
print(f"Đang tải dữ liệu từ: {CORPUS_FILE}...")
corpus_df = pd.read_csv(CORPUS_FILE)
documents = corpus_df['Question'].tolist()
print(f"Đã tải {len(documents)} tài liệu.")

# --- 3. Tách từ (Tokenize) ---
# Đây là bước quan trọng cho BM25 tiếng Việt
print("Đang tách từ (tokenize) toàn bộ 12,060 tài liệu... (Việc này có thể mất vài phút)")

def tokenize_vietnamese(text):
    # Dùng ViTokenizer để tách từ
    return ViTokenizer.tokenize(text).split()

# Áp dụng tách từ cho tất cả tài liệu
tokenized_corpus = [tokenize_vietnamese(doc) for doc in documents]
print("Đã tách từ xong.")

# --- 4. Huấn luyện BM25 ---
print("Đang huấn luyện (fit) BM25 Index...")
bm25 = BM25Okapi(tokenized_corpus)
print("Huấn luyện BM25 hoàn tất.")

# --- 5. Lưu Index ra File ---
print(f"Đang lưu BM25 Index ra file: {OUTPUT_BM25_FILE}...")
# Dùng pickle để lưu đối tượng bm25 đã huấn luyện
with open(OUTPUT_BM25_FILE, 'wb') as f:
    pickle.dump(bm25, f)

print("\n--- HOÀN TẤT ---")
print(f"Đã tạo và lưu file BM25 index tại: {OUTPUT_BM25_FILE}")

In [ ]:
import pandas as pd
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer
import torch
from tqdm.auto import tqdm
import os
import pickle
from pyvi import ViTokenizer

print("--- SCRIPT ĐÁNH GIÁ (So sánh Hybrid Search) ---")

# --- 1. Cấu hình & Tải Mọi thứ ---
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"--- Đang chạy trên: {device} ---")

# Đường dẫn (Giờ chúng ta chỉ cần 1 mô hình, 2 index)
MODEL_PATH = './models/triplet-finetuned-model-v2/'
INDEX_FAISS_PATH = './artifacts/medical_index_B.faiss'
INDEX_BM25_PATH = './artifacts/medical_index_B.bm25'
CORPUS_FILE = 'corpus_with_id.csv'
TEST_SET_FILE = 'test_set_new.csv'
TOP_K_SEARCH = 10 # K cuối cùng để đánh giá (Recall@10)
K_RETRIEVE_HYBRID = 50 # Số lượng lấy về cho Hybrid

# --- 2. Tải Mô hình & Index ---
print("Đang tải Model B (Bi-Encoder)...")
model_B = SentenceTransformer(MODEL_PATH, device=device)

print("Đang tải FAISS Index...")
index_faiss = faiss.read_index(INDEX_FAISS_PATH)

print("Đang tải BM25 Index...")
with open(INDEX_BM25_PATH, 'rb') as f:
    index_bm25 = pickle.load(f)

# --- 3. Tải Dữ liệu (Corpus và Test Set) ---
print("Đang tải dữ liệu...")
corpus_df = pd.read_csv(CORPUS_FILE)
corpus_doc_ids = corpus_df['doc_id'].tolist()
index_to_id_map = {i: doc_id for i, doc_id in enumerate(corpus_doc_ids)}
id_to_text_map = pd.Series(corpus_df.Question.values, index=corpus_df.doc_id).to_dict()

test_set_df = pd.read_csv(TEST_SET_FILE)
test_set_df = test_set_df.dropna(subset=['query', 'ground_truth_doc_id'])
test_set_df['ground_truth_doc_id'] = test_set_df['ground_truth_doc_id'].astype(int)
print(f"Đã tải {len(test_set_df)} câu test.")

# --- 4. Các Hàm Logic Truy xuất ---

def tokenize_vietnamese(text):
    return ViTokenizer.tokenize(text).split()

def retrieve_dense_only(query_vec, k):
    """Chỉ tìm bằng FAISS (Baseline)"""
    D, I = index_faiss.search(query_vec.astype(np.float32), k)
    return [index_to_id_map[i] for i in I[0]]

def retrieve_hybrid(query_text, query_vec, k_final):
    """Tìm kiếm lai (FAISS + BM25 + RRF)"""
    K_RETRIEVE = K_RETRIEVE_HYBRID
    RRF_K = 60
    
    # A: Dense Search (đã có query_vec)
    D, I_faiss = index_faiss.search(query_vec.astype(np.float32), K_RETRIEVE)
    
    # B: Sparse Search
    tokenized_query = tokenize_vietnamese(query_text)
    bm25_scores = index_bm25.get_scores(tokenized_query)
    I_bm25 = np.argsort(bm25_scores)[::-1][:K_RETRIEVE]
    
    # C: RRF Fusion
    rrf_scores = {}
    for rank, faiss_index in enumerate(I_faiss[0]):
        doc_id = index_to_id_map[faiss_index]
        rrf_scores[doc_id] = rrf_scores.get(doc_id, 0.0) + 1.0 / (RRF_K + rank + 1)
        
    for rank, bm25_index in enumerate(I_bm25):
        doc_id = index_to_id_map[bm25_index]
        rrf_scores[doc_id] = rrf_scores.get(doc_id, 0.0) + 1.0 / (RRF_K + rank + 1)
        
    # D: Xếp hạng cuối
    sorted_hybrid_results = sorted(rrf_scores.items(), key=lambda item: item[1], reverse=True)
    return [doc_id for doc_id, score in sorted_hybrid_results[:k_final]]

# --- 5. Hàm Tính toán (Giữ nguyên) ---
def calculate_metrics(retrieved_doc_ids, ground_truth_id, k_values=[1, 3, 5, 10]):
    # (Code hàm calculate_metrics giữ nguyên như cũ)
    metrics = {}
    retrieved_set = set(retrieved_doc_ids)
    for k in k_values:
        top_k_set = set(retrieved_doc_ids[:k])
        is_hit = 1.0 if ground_truth_id in top_k_set else 0.0
        metrics[f'Recall@{k}'] = is_hit
    reciprocal_rank = 0.0
    for rank, doc_id in enumerate(retrieved_doc_ids):
        if doc_id == ground_truth_id:
            reciprocal_rank = 1.0 / (rank + 1)
            break
    metrics['MRR'] = reciprocal_rank
    return metrics

# --- 6. Vòng lặp Đánh giá ---
print("\n--- BẮT ĐẦU VÒNG LẶP ĐÁNH GIÁ (So sánh Hybrid) ---")

results_baseline = []
results_hybrid = []

for row in tqdm(test_set_df.itertuples(), total=len(test_set_df), desc="Đánh giá Golden Set"):
    query = row.query
    ground_truth_id = int(row.ground_truth_doc_id)
    
    # Chuẩn bị Query Vector (dùng chung)
    query_vec = model_B.encode([query])
    faiss.normalize_L2(query_vec)
    
    # Chạy Baseline (Chỉ Dense)
    retrieved_baseline = retrieve_dense_only(query_vec, TOP_K_SEARCH)
    metrics_baseline = calculate_metrics(retrieved_baseline, ground_truth_id)
    results_baseline.append(metrics_baseline)
    
    # Chạy Challenger (Hybrid)
    retrieved_hybrid = retrieve_hybrid(query, query_vec, TOP_K_SEARCH)
    metrics_hybrid = calculate_metrics(retrieved_hybrid, ground_truth_id)
    results_hybrid.append(metrics_hybrid)

print("--- Đánh giá hoàn tất ---")

# --- 7. Hiển thị Kết quả ---
metrics_baseline_mean = pd.DataFrame(results_baseline).mean()
metrics_hybrid_mean = pd.DataFrame(results_hybrid).mean()

comparison_df = pd.DataFrame({
    "Baseline (Dense Only)": metrics_baseline_mean,
    "Challenger (Hybrid Search)": metrics_hybrid_mean
})
comparison_df['Improvement (%)'] = ((comparison_df['Challenger (Hybrid Search)'] - comparison_df['Baseline (Dense Only)']) / comparison_df['Baseline (Dense Only)']) * 100

print("\n--- BẢNG SO SÁNH HIỆU SUẤT (Baseline vs Hybrid) ---")
print(comparison_df.to_markdown(floatfmt=".4f"))

# --- Quyết định ---
recall_5_baseline = comparison_df.loc['Recall@5', 'Baseline (Dense Only)']
recall_5_hybrid = comparison_df.loc['Recall@5', 'Challenger (Hybrid Search)']

if recall_5_hybrid > recall_5_baseline:
    print(f"\nTHÀNH CÔNG! Hybrid Search (Recall@5: {recall_5_hybrid:.4f}) tốt hơn Baseline (Recall@5: {recall_5_baseline:.4f}).")
else:
    print(f"\nTHẤT BẠI. Hybrid Search không cải thiện được hiệu suất.")

In [ ]:
import pandas as pd
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer, CrossEncoder # Thêm CrossEncoder
import torch
from tqdm.auto import tqdm
import os
import pickle
from pyvi import ViTokenizer
import gc # Để giải phóng VRAM

print("--- SCRIPT ĐÁNH GIÁ (So sánh Hybrid vs Hybrid + Re-Ranker) ---")

# --- 1. Cấu hình & Tải ---
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"--- Đang chạy trên: {device} ---")

# --- Đường dẫn ---
MODEL_PATH = './models/triplet-finetuned-model-v2/'
INDEX_FAISS_PATH = './artifacts/medical_index_B.faiss'
INDEX_BM25_PATH = './artifacts/medical_index_B.bm25'
CORPUS_FILE = 'corpus_with_id.csv'
TEST_SET_FILE = 'test_set_new.csv'
RERANKER_MODEL_PATH = 'cross-encoder/mmarco-mMiniLMv2-L12-H384-v1'

# --- Tham số Đánh giá ---
TOP_K_FINAL = 10      # Số lượng kết quả cuối cùng để tính metrics (Recall@10)
K_RETRIEVE_HYBRID = 50 # Số lượng ứng cử viên Hybrid Search lấy về để cho Re-Ranker

# --- 2. Tải Tất cả Mô hình & Index ---
print("Đang tải Model B (Bi-Encoder)...")
model_B = SentenceTransformer(MODEL_PATH, device=device)

print(f"Đang tải Model Re-Ranker (Cross-Encoder)...")
# Tải Cross-Encoder, cũng đẩy lên GPU
cross_encoder = CrossEncoder(RERANKER_MODEL_PATH, max_length=512, device=device)

print("Đang tải FAISS Index...")
index_faiss = faiss.read_index(INDEX_FAISS_PATH)

print("Đang tải BM25 Index...")
with open(INDEX_BM25_PATH, 'rb') as f:
    index_bm25 = pickle.load(f)

# --- 3. Tải Dữ liệu ---
print("Đang tải dữ liệu...")
corpus_df = pd.read_csv(CORPUS_FILE)
index_to_id_map = {i: doc_id for i, doc_id in enumerate(corpus_df['doc_id'].tolist())}
id_to_text_map = pd.Series(corpus_df.Question.values, index=corpus_df.doc_id).to_dict()

test_set_df = pd.read_csv(TEST_SET_FILE)
test_set_df = test_set_df.dropna(subset=['query', 'ground_truth_doc_id'])
test_set_df['ground_truth_doc_id'] = test_set_df['ground_truth_doc_id'].astype(int)
print(f"Đã tải {len(test_set_df)} câu test.")

# --- 4. Các Hàm Logic Truy xuất & Xếp hạng ---
def tokenize_vietnamese(text):
    return ViTokenizer.tokenize(text).split()

def retrieve_hybrid_candidates(query_text, query_vec, k_retrieve):
    """
    Chạy Hybrid Search (FAISS + BM25 + RRF) để lấy về K ứng cử viên.
    """
    RRF_K = 60
    
    # A: Dense Search
    D, I_faiss = index_faiss.search(query_vec.astype(np.float32), k_retrieve)
    
    # B: Sparse Search
    tokenized_query = tokenize_vietnamese(query_text)
    bm25_scores = index_bm25.get_scores(tokenized_query)
    I_bm25 = np.argsort(bm25_scores)[::-1][:k_retrieve]
    
    # C: RRF Fusion
    rrf_scores = {}
    for rank, faiss_index in enumerate(I_faiss[0]):
        doc_id = index_to_id_map[faiss_index]
        rrf_scores[doc_id] = rrf_scores.get(doc_id, 0.0) + 1.0 / (RRF_K + rank + 1)
        
    for rank, bm25_index in enumerate(I_bm25):
        doc_id = index_to_id_map[bm25_index]
        rrf_scores[doc_id] = rrf_scores.get(doc_id, 0.0) + 1.0 / (RRF_K + rank + 1)
        
    # Sắp xếp
    sorted_hybrid_results = sorted(rrf_scores.items(), key=lambda item: item[1], reverse=True)
    return [doc_id for doc_id, score in sorted_hybrid_results[:k_retrieve]]

def rerank_candidates(query_text, candidate_doc_ids):
    """
    Chạy Cross-Encoder để chấm điểm lại danh sách ứng cử viên.
    """
    # 1. Tạo các cặp (query, document_text)
    pairs = []
    for doc_id in candidate_doc_ids:
        doc_text = id_to_text_map.get(doc_id, "") # Lấy text từ ID
        pairs.append([query_text, doc_text])
        
    # 2. Chấm điểm (Đây là bước "chậm" nhưng chính xác)
    # Gói gọn trong 'with torch.no_grad()' để tiết kiệm VRAM
    with torch.no_grad():
        scores = cross_encoder.predict(pairs, show_progress_bar=False, convert_to_tensor=True)
    
    # 3. Kết hợp ID và điểm số mới
    reranked_results = list(zip(candidate_doc_ids, scores))
    
    # 4. Sắp xếp theo điểm số mới (cao -> thấp)
    reranked_results.sort(key=lambda x: x[1], reverse=True)
    
    # Trả về danh sách doc_id đã được xếp hạng lại
    return [doc_id for doc_id, score in reranked_results]

# --- 5. Hàm Tính toán (Giữ nguyên) ---
def calculate_metrics(retrieved_doc_ids, ground_truth_id, k_values=[1, 3, 5, 10]):
    # (Code hàm calculate_metrics giữ nguyên như cũ)
    metrics = {}
    retrieved_set = set(retrieved_doc_ids)
    for k in k_values:
        top_k_set = set(retrieved_doc_ids[:k])
        is_hit = 1.0 if ground_truth_id in top_k_set else 0.0
        metrics[f'Recall@{k}'] = is_hit
    reciprocal_rank = 0.0
    for rank, doc_id in enumerate(retrieved_doc_ids):
        if doc_id == ground_truth_id:
            reciprocal_rank = 1.0 / (rank + 1)
            break
    metrics['MRR'] = reciprocal_rank
    return metrics

# --- 6. Vòng lặp Đánh giá ---
print("\n--- BẮT ĐẦU VÒNG LẶP ĐÁNH GIÁ (So sánh Re-Ranker) ---")

results_hybrid_only = []
results_reranked = []

for row in tqdm(test_set_df.itertuples(), total=len(test_set_df), desc="Đánh giá Golden Set"):
    query = row.query
    ground_truth_id = int(row.ground_truth_doc_id)
    
    # Chuẩn bị Query Vector (dùng chung)
    query_vec = model_B.encode([query])
    faiss.normalize_L2(query_vec)
    
    # --- Chạy 2 chiến lược ---
    
    # 1. Baseline (Hybrid Only): Lấy Top 50, nhưng chỉ đánh giá Top 10
    hybrid_candidates = retrieve_hybrid_candidates(query, query_vec, K_RETRIEVE_HYBRID)
    metrics_baseline = calculate_metrics(hybrid_candidates, ground_truth_id, k_values=[1, 3, 5, 10])
    results_hybrid_only.append(metrics_baseline)
    
    # 2. Challenger (Hybrid + Re-Ranker): Lấy Top 50, Sắp xếp lại, Đánh giá Top 10
    reranked_doc_ids = rerank_candidates(query, hybrid_candidates)
    metrics_reranked = calculate_metrics(reranked_doc_ids, ground_truth_id, k_values=[1, 3, 5, 10])
    results_reranked.append(metrics_reranked)

print("--- Đánh giá hoàn tất ---")

# --- 7. Hiển thị Kết quả ---
metrics_baseline_mean = pd.DataFrame(results_hybrid_only).mean()
metrics_reranked_mean = pd.DataFrame(results_reranked).mean()

comparison_df = pd.DataFrame({
    "Baseline (Hybrid Only)": metrics_baseline_mean,
    "Challenger (Hybrid + Re-Ranker)": metrics_reranked_mean
})
comparison_df['Improvement (%)'] = ((comparison_df['Challenger (Hybrid + Re-Ranker)'] - comparison_df['Baseline (Hybrid Only)']) / comparison_df['Baseline (Hybrid Only)']) * 100

print("\n--- BẢNG SO SÁNH HIỆU SUẤT (Hybrid vs. Re-Ranker) ---")
print(comparison_df.to_markdown(floatfmt=".4f"))

# --- Quyết định ---
mrr_baseline = comparison_df.loc['MRR', 'Baseline (Hybrid Only)']
mrr_reranked = comparison_df.loc['MRR', 'Challenger (Hybrid + Re-Ranker)']

if mrr_reranked > mrr_baseline:
    print(f"\nTHÀNH CÔNG! Re-Ranker (MRR: {mrr_reranked:.4f}) tốt hơn Baseline (MRR: {mrr_baseline:.4f}).")
else:
    print(f"\nTHẤT BẠI. Re-Ranker không cải thiện được hiệu suất.")

f:\NLP_AI\.project\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


--- SCRIPT ĐÁNH GIÁ (So sánh Hybrid vs Hybrid + Re-Ranker) ---
--- Đang chạy trên: cuda ---
Đang tải Model B (Bi-Encoder)...
Đang tải Model Re-Ranker (Cross-Encoder)...


f:\NLP_AI\.project\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\ADMIN\.cache\huggingface\hub\models--cross-encoder--mmarco-mMiniLMv2-L12-H384-v1. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HT

Đang tải FAISS Index...
Đang tải BM25 Index...
Đang tải dữ liệu...
Đã tải 105 câu test.

--- BẮT ĐẦU VÒNG LẶP ĐÁNH GIÁ (So sánh Re-Ranker) ---


Đánh giá Golden Set: 100%|██████████| 105/105 [00:14<00:00,  7.39it/s]

--- Đánh giá hoàn tất ---

--- BẢNG SO SÁNH HIỆU SUẤT (Hybrid vs. Re-Ranker) ---
|           |   Baseline (Hybrid Only) |   Challenger (Hybrid + Re-Ranker) |   Improvement (%) |
|:----------|-------------------------:|----------------------------------:|------------------:|
| Recall@1  |                   0.0667 |                            0.0667 |            0.0000 |
| Recall@3  |                   0.0762 |                            0.1143 |           50.0000 |
| Recall@5  |                   0.1048 |                            0.1333 |           27.2727 |
| Recall@10 |                   0.1619 |                            0.2000 |           23.5294 |
| MRR       |                   0.0910 |                            0.1076 |           18.2707 |

✅ THÀNH CÔNG! Re-Ranker (MRR: 0.1076) tốt hơn Baseline (MRR: 0.0910).
